# Zonal statistics

Summarise raster values **within each vector polygon** — the classic "average elevation per
watershed" operation. `ds.zonal_stats(polygons, stats=(...))` returns a table with one row
per polygon and one column per requested statistic.

## Setup

In [1]:
import os

os.environ['MPLBACKEND'] = 'Agg'  # never trigger an interactive backend

import tempfile
from pathlib import Path

import numpy as np


def _find_data():
    for base in [Path.cwd(), *Path.cwd().parents]:
        cand = base / 'tests' / 'data'
        if cand.is_dir():
            return cand.resolve()
    raise FileNotFoundError('Could not locate tests/data from ' + str(Path.cwd()))


DATA = _find_data()
WORK = Path(tempfile.mkdtemp(prefix='pyramids-ops-'))
DATA.is_dir(), WORK.is_dir()

(True, True)

In [2]:
from pyramids.dataset import Dataset
from pyramids.feature import FeatureCollection

ds = Dataset.read_file(str(DATA / 'acc4000.tif'))
zones = FeatureCollection.read_file(str(DATA / 'coello_polygons.geojson'))
len(zones), zones.epsg

2026-06-08 23:01:59 | INFO | pyramids.base.config | Logging is configured.


(4, 32618)

## Compute per-zone statistics

Each polygon is rasterised onto the raster grid, then the covered cells are reduced.

In [3]:
stats = ds.zonal_stats(zones, stats=('mean', 'min', 'max', 'sum', 'count'))
stats

C:\gdrive\algorithms\gis\pyramids\.claude\worktrees\operation-notebooks\.pixi\envs\dev\Lib\site-packages\osgeo\ogr.py:7515: RuntimeWarning: DeprecationWarning: 'Memory' driver is deprecated since GDAL 3.11. Use 'MEM' onwards. Further messages of this type will be suppressed.
  return _ogr.GetDriverByName(*args)


,mean,min,max,sum,count
0,0.000000,0.0,0.0,0.0,1.0
1,0.000000,0.0,0.0,0.0,2.0
2,2.307692,0.0,11.0,30.0,13.0
3,18.066667,0.0,63.0,271.0,15.0


## Notes

- `band=` selects which band to summarise (default 0).
- The polygons may be in any CRS — they are reprojected to the raster's CRS internally.
- See also: [Crop & mask](crop-mask.ipynb), [Extract at points](extract-at-points.ipynb).